# Importance XGB

# Setup

## Imports

In [ ]:
import pathlib
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
    root_mean_squared_error,
)

from thesis_project import (
    config as global_config,
)
from thesis_project.importance_xgb import config, data_paths

## Paths

In [ ]:
TICKER = "fbtp"
# NOTE: Use "notebook-name" if needed
NOTEBOOK = "importance-xgb"

# Lobs directories
LOB_DIR = global_config.RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "futures"
LOB_CTD_DIR = LOB_DIR / "ctd"

# Output directories
if NOTEBOOK is not None:
    OUT_FIG_DIR = global_config.FIG_DIR / TICKER / NOTEBOOK
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = global_config.TAB_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    # NOTE: Create a purpose specific directory if needed
    # OUT_INT_DIR = config.INT_DIR / TICKER / NOTEBOOK
    # OUT_INT_DIR.mkdir(parents=True, exist_ok=True)
    # OUT_PRO_DIR = config.PRO_DIR / TICKER / NOTEBOOK
    # OUT_PRO_DIR.mkdir(parents=True, exist_ok=True)

## Settings

In [ ]:
# Matplotlib global settings
# config.MPL contains a personalized default configuration for matplotlib
plt.rcdefaults()
plt.style.use("seaborn-v0_8-paper")

In [ ]:
# Warnings settings
warnings.simplefilter("always", UserWarning)

# Main

## Load data

In [ ]:
CLS_LABELS = ["Low", "Medium", "High"]
CLS_DICT = {k: v for k, v in enumerate(CLS_LABELS)}

run = "003"
dir = f"run_{run}"
base_path = global_config.INT_DIR / TICKER / "importance-xgb" / "runs"
path = base_path / dir

with open(path / "config.yaml") as f:
    config_dict = yaml.safe_load(f)
exp_config_obj = config.AnalysisConfig(**config_dict)

with open(path / "data_paths.yaml") as f:
    data_paths_dict = yaml.safe_load(f)
    target = next((name, pathlib.Path(path)) for name, path in data_paths_dict["target"].items())

exp_data_paths = data_paths.DataPathsConfig(
    ticker=TICKER, target=target, features=data_paths_dict["features"]
)

with open(path / "metadata.yaml") as f:
    exp_metadata = yaml.safe_load(f)

importance_xgb_dict = {}
for seed in exp_metadata["seeds"]:
    importance_xgb_dict[seed] = pd.read_csv(path / f"importance_xgb_seed_{seed}.csv")

In [ ]:
import sqlite3

conn = sqlite3.connect(path / "optuna.db")
cur = conn.cursor()

cur.execute(
    """
UPDATE studies
SET study_name = ?
WHERE study_name = ?
""",
    (f"run_{run}", "run_002"),
)

conn.commit()
conn.close()

## Metrics

In [ ]:
y_true = pd.read_parquet(path / "y_cls_test.parquet").iloc[:, 0]

y_proba_dict = {}
y_pred_dict = {}
metrics = {}
for seed in exp_metadata["seeds"]:
    y_proba_dict[seed] = pd.read_parquet(path / f"y_cls_proba_test_seed_{seed}.parquet")
    y_pred_dict[seed] = np.argmax(y_proba_dict[seed], axis=1)

for seed in exp_metadata["seeds"]:
    metrics[seed] = {
        "logloss": log_loss(y_true, y_proba_dict[seed], labels=range(exp_config_obj.n_quantile)),
        "accuracy": accuracy_score(y_true, y_pred_dict[seed]),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred_dict[seed]),
        "precision": precision_score(y_true, y_pred_dict[seed], average="macro"),
        "recall": recall_score(y_true, y_pred_dict[seed], average="macro"),
        "f1": f1_score(y_true, y_pred_dict[seed], average="macro"),
        "auc": roc_auc_score(y_true, y_proba_dict[seed], average="macro", multi_class="ovo"),
        "mcc": matthews_corrcoef(y_true, y_pred_dict[seed]),
    }

df = pd.DataFrame.from_dict(metrics, orient="index")
df.index.name = "seed"
df.columns.name = "metrics"
display(df)

# df.to_csv(OUT_TAB_DIR / f"{dir}_models_performances.csv")

df = pd.concat(
    [
        df.mean().to_frame().T.rename(index={0: "mean"}),
        df.std().to_frame().T.rename(index={0: "std"}),
    ]
)
display(df)

In [ ]:
y_true = pd.read_parquet(path / "y_test.parquet").iloc[:, 0]

y_pred_dict = {}
metrics = {}
for seed in exp_metadata["seeds"]:
    y_pred_dict[seed] = pd.read_parquet(path / f"y_pred_test_seed_{seed}.parquet")

for seed in exp_metadata["seeds"]:
    metrics[seed] = {
        "rmse": root_mean_squared_error(y_true, y_pred_dict[seed]),
        "r2": r2_score(y_true, y_pred_dict[seed]),
    }

df = pd.DataFrame.from_dict(metrics, orient="index")
df.index.name = "seed"
df.columns.name = "metrics"
display(df)

# df.to_csv(OUT_TAB_DIR / f"{dir}_models_performances.csv")

df = pd.concat(
    [
        df.mean().to_frame().T.rename(index={0: "mean"}),
        df.std().to_frame().T.rename(index={0: "std"}),
    ]
)
display(df)

## Confusion Matrix

In [ ]:
def plot_cm(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred, normalize="true")

    fig, ax = plt.subplots(figsize=(6, 6))

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, values_format=".2f", cmap="Blues")

    ax.set_title(f"Confusion Matrix \nSeed: {title}", fontsize=16)
    ax.set_xlabel("Predicted label", fontsize=12)
    ax.set_ylabel("True label", fontsize=12)

    fig.tight_layout()
    plt.show()


def plot_mean_cm(y_true, y_pred_dict, labels):
    cms = []

    for _, y_pred in y_pred_dict.items():
        cm = confusion_matrix(y_true, y_pred, labels=range(len(labels)), normalize="true")
        cms.append(cm)

    mean_cm = np.mean(cms, axis=0)

    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=mean_cm, display_labels=labels)
    disp.plot(ax=ax, values_format=".2f", cmap="Blues", colorbar=True)

    ax.set_title("Mean Confusion Matrix Across Seeds")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")

    fig.tight_layout()
    plt.show()

In [ ]:
for seed, y_pred in y_pred_dict.items():
    plot_cm(y_true, y_pred, CLS_LABELS, seed)

In [ ]:
plot_mean_cm(y_true, y_pred_dict, CLS_LABELS)

## Model importance

In [ ]:
def importance_xgb_ranking(importance_xgb, n_features, metric, seed):
    data = (
        importance_xgb.sort_values(by=metric, ascending=False)
        .reset_index(drop=True)
        .head(n_features)
    )
    data.index += 1
    data.index.name = "rank"
    cols = ["feature", metric]
    cols = cols + [c for c in data.columns if c not in cols]
    data = data[cols]
    print(f"Seed: {seed}")
    display(data)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(data["feature"], data[metric])

    ax.invert_yaxis()
    ax.set_xlabel(metric.title())
    ax.set_ylabel("Feature")
    ax.set_title(f"Top {n_features} features by {metric} \nSeed: {seed}")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.tight_layout()
    plt.show()


def importance_xgb_mean_ranking(importance_xgb_dict, n_features, metric):
    dfs = []

    for seed, df in importance_xgb_dict.items():
        tmp = df[["feature", metric]].copy()
        tmp["seed"] = seed
        dfs.append(tmp)

    all_importances = pd.concat(dfs, ignore_index=True)

    summary = (
        all_importances.groupby("feature")[metric]
        .agg(mean="mean", std="std")
        .fillna(0)
        .sort_values("mean", ascending=False)
        .head(n_features)
        .reset_index()
    )

    summary.index += 1
    summary.index.name = "rank"

    # display(summary)

    fig, ax = plt.subplots(figsize=(12, 8))

    ax.barh(
        summary["feature"],
        summary["mean"],
        xerr=summary["std"],
        capsize=3,
    )

    ax.invert_yaxis()
    ax.set_xlabel(f"Mean {metric.title()}")
    ax.set_ylabel("Feature")
    ax.set_title(f"Top {n_features} Features by Mean {metric.title()}")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.tight_layout()
    plt.show()


def feature_rankings(importance_xgb_dict, metric, show_rank=False):
    rankings = {}

    if show_rank:
        key = "rank"
    else:
        key = metric

    for seed, df in importance_xgb_dict.items():
        ranked = df.sort_values(metric, ascending=False).reset_index(drop=True)
        ranked["rank"] = range(1, len(ranked) + 1)
        rankings[seed] = ranked.set_index("feature")[key]

    rankings = pd.DataFrame(rankings)
    rankings.columns.name = "seed"

    rankings["mean"] = rankings.mean(axis=1)
    rankings["std"] = rankings.std(axis=1)

    return rankings.sort_values("mean", ascending=show_rank)

In [ ]:
n = 20
metric = "gain"
for seed, importance_xgb in importance_xgb_dict.items():
    importance_xgb_ranking(importance_xgb, n, metric, seed)

In [ ]:
n = 25
metric = "gain"
importance_xgb_mean_ranking(importance_xgb_dict, n, metric)

In [ ]:
metric = "gain"
n = 50
show_rank = True
df = feature_rankings(importance_xgb_dict, metric, show_rank)
display(df.head(n))

## Shap Importance

In [ ]:
import copy

import shap

n = 11
for cls in range(exp_config_obj.n_quantile):
    explainers = [
        joblib.load(path / f"importance_shap_seed_{seed}.pkl") for seed in exp_metadata["seeds"]
    ]

    # Average SHAP values across seeds
    mean_exp = copy.deepcopy(explainers[0])
    mean_exp.values = np.mean([exp.values for exp in explainers], axis=0)

    # If base values vary slightly, average those too
    if mean_exp.base_values is not None:
        mean_exp.base_values = np.mean([exp.base_values for exp in explainers], axis=0)

    shap.plots.beeswarm(mean_exp[:, :, cls], max_display=n, show=False)
    plt.title(f"Average over {len(explainers)} seeds\nQuantile: {CLS_DICT[cls]}")
    plt.tight_layout()
    plt.show()

In [ ]:
import copy

import shap

n = 11
explainers = [
    joblib.load(path / f"importance_shap_seed_{seed}.pkl") for seed in exp_metadata["seeds"]
]

# Average SHAP values across seeds
mean_exp = copy.deepcopy(explainers[0])
mean_exp.values = np.mean([exp.values for exp in explainers], axis=0)

# If base values vary slightly, average those too
if mean_exp.base_values is not None:
    mean_exp.base_values = np.mean([exp.base_values for exp in explainers], axis=0)

shap.plots.beeswarm(mean_exp, max_display=n, show=False)
plt.title(f"Average over {len(explainers)} seeds\nQuantile: {CLS_DICT[cls]}")
plt.tight_layout()
plt.show()

In [ ]:
def keys_transform(config_obj: config.AnalysisConfig):
    keys = []
    for transform in config_obj.feature_transforms:
        if transform["name"] == "delta":
            for delta in transform["params"]["deltas"]:
                keys.append(f"delta_{delta}s")

        if transform["name"] == "rolling":
            for stat in transform["params"]["stats"]:
                for window in transform["params"]["windows"]:
                    keys.append(f"roll_{stat}_{window}s")

        if transform["name"] == "ratio":
            for ref in transform["params"]["references"]:
                keys.append(f"{ref}_ratio")

    return keys

In [ ]:
for explainer in explainers:
    all_features = explainer.feature_names
    # keys = list(exp_data_paths.features.keys())
    keys = keys_transform(exp_config_obj)

    groups = {key: [i for i, feature in enumerate(all_features) if key in feature] for key in keys}

    data = np.column_stack([explainer.data[:, idxs].sum(axis=1) for idxs in groups.values()])

    n = 11
    for cls in range(exp_config_obj.n_quantile):
        values = np.column_stack(
            [explainer.values[:, idxs, cls].sum(axis=1) for idxs in groups.values()]
        )

        grouped = shap.Explanation(values=values, data=None, feature_names=keys)

        size = 14
        # shap.plots.beeswarm(grouped, max_display=n, show=False)
        # plt.title(f"Grouped SHAP Beeswarm - {CLS_DICT[cls]}", size=size)
        # plt.tight_layout()
        # plt.show()

        shap.plots.bar(grouped, max_display=n, show=False)
        plt.title(f"Grouped SHAP Barplot - {CLS_DICT[cls]}", size=size)
        plt.tight_layout()
        plt.show()

In [ ]:
for explainer in explainers:
    all_features = explainer.feature_names
    # keys = list(exp_data_paths.features.keys())
    keys = keys_transform(exp_config_obj)

    groups = {key: [i for i, feature in enumerate(all_features) if key in feature] for key in keys}

    data = np.column_stack([explainer.data[:, idxs].sum(axis=1) for idxs in groups.values()])

    n = 11
    values = np.column_stack([explainer.values[:, idxs].sum(axis=1) for idxs in groups.values()])

    grouped = shap.Explanation(values=values, data=None, feature_names=keys)

    size = 14
    # shap.plots.beeswarm(grouped, max_display=n, show=False)
    # plt.title(f"Grouped SHAP Beeswarm - {CLS_DICT[cls]}", size=size)
    # plt.tight_layout()
    # plt.show()

    shap.plots.bar(grouped, max_display=n, show=False)
    plt.title("Grouped SHAP Barplot", size=size)
    plt.tight_layout()
    plt.show()

## Model

In [ ]:
import optuna
from xgboost import XGBRegressor

params_dict = {}
for seed in exp_metadata["seeds"]:
    model = XGBRegressor()
    model.load_model(path / f"model_seed_{seed}.ubj")
    study = optuna.load_study(study_name=f"run_{run}", storage=f"sqlite:///{path / 'optuna.db'}")
    params = study.best_trial.params
    params["n_trees"] = model.get_booster().num_boosted_rounds()
    params_dict[seed] = params

df = pd.DataFrame().from_dict(params_dict, orient="index")
df.index.name = "seed"
df.columns.name = "params"

display(df)